In [1]:
"""
FINAL TEST (Advanced Machine Learning) group C

Upload files with solutions and plots to MS TEAMS: test.py (or test.ipynp), PLOT1.pdf,
PLOT2.pdf, PLOT3.pdf, PLOT4.pdf (jpg format can be also used).
Deadline is: 12 June, 3.50 PM.
DO NOT use archive file format such as zip!
Uploading files after the deadline or using the wrong format will result in reduced points
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from scipy.interpolate import UnivariateSpline

np.random.seed(42)

# =============================================================================
# 1  Task 1 (2 points)
#
# Generate artifical data with as follows:
# - Generate features X_1, ..., X_10 from N(0, sd=1).
# - Generate binary class variable such that P(Y=1|X=x) = sigma(x^T mu),
#   where sigma is logistic sigmoid function and mu = (2,...,2) is a vector
#   of length 10.
# - Generate training data D_train = {(x_i, y_i) : i=1,...,n} of size n=1000,
#   where x_i is 10-dimensional feature vector for i-th observation and y_i
#   is the value of class variable for i-th observation.
# - In addition generate testing data D_test of size n=1000 using the above
#   scheme.
# =============================================================================

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

n = 1000
p = 10
mu = np.full(p, 2.0)

# Training data
X_train = np.random.normal(0, 1, (n, p))
prob_train = sigmoid(X_train @ mu)
y_train = (np.random.rand(n) < prob_train).astype(int)

# Testing data
X_test = np.random.normal(0, 1, (n, p))
prob_test = sigmoid(X_test @ mu)
y_test = (np.random.rand(n) < prob_test).astype(int)

print("Task 1 done.")
print(f"  Training set: {X_train.shape}, class balance: {y_train.mean():.3f}")
print(f"  Testing set:  {X_test.shape}, class balance: {y_test.mean():.3f}")


# =============================================================================
# 2  Task 2 (7 points)
#
# Assume that our model predicts P(Y=1|X=x) using logistic sigmoid function
# sigma(x^T theta) and the optimal values of parameters theta=(theta_1,...,
# theta_10) are found by minimizing the following weighted risk function on
# training data:
#
#   R(theta) = -1/n * sum_{i=1}^{n} w1*y_i*log[sigma(x_i^T theta)]
#              -1/n * sum_{i=1}^{n} w2*(1-y_i)*log[1-sigma(x_i^T theta)]
#              + 0.01 * sum_{j=1}^{10} theta_j^2
#
# where w1=2 and w2=1 are weights. To make a prediction for new observation
# we apply the rule y_hat=1 when sigma(x^T theta_hat) > 0.5.
#
# - Implement from scratch (1) and run gradient descent (GD) algorithm with
#   learning rate lr=0.1 and number of epochs=1000. Take as the initialization
#   point theta=(0,...,0).
#   (1) It is not allowed to use ready-made implementations of the GD
#       algorithm, e.g. the one from the python library.
# - Generete plot showing how the value of the risk changes for subsequent
#   iterations. Save the results in file PLOT1.pdf.
# - Generete a plot showing how the accuracy of the model calculated on
#   training data changes for subsequent iterations. Save the results in file
#   PLOT2.pdf. Consider also the prediction for the initialization point.
# - Generete a plot showing how the accuracy of the model calculated on
#   testing data changes for subsequent iterations. Save the results in file
#   PLOT3.pdf. Consider also the prediction for the initialization point.
# =============================================================================

w1, w2 = 2.0, 1.0
lr = 0.1
epochs = 1000
lam = 0.01  # regularization coefficient

def risk(theta, X, y):
    z = X @ theta
    p_hat = sigmoid(z)
    # clip to avoid log(0)
    p_hat = np.clip(p_hat, 1e-12, 1 - 1e-12)
    log_loss = (-w1 * y * np.log(p_hat) - w2 * (1 - y) * np.log(1 - p_hat)).mean()
    reg = lam * np.sum(theta ** 2)
    return log_loss + reg

def gradient(theta, X, y):
    n_local = X.shape[0]
    z = X @ theta
    p_hat = sigmoid(z)
    # gradient of weighted log-loss part
    residual = -w1 * y * (1 - p_hat) + w2 * (1 - y) * p_hat
    grad = (X.T @ residual) / n_local + 2 * lam * theta
    return grad

def accuracy(theta, X, y):
    preds = (sigmoid(X @ theta) > 0.5).astype(int)
    return (preds == y).mean()

theta = np.zeros(p)

risks = [risk(theta, X_train, y_train)]
acc_train = [accuracy(theta, X_train, y_train)]
acc_test = [accuracy(theta, X_test, y_test)]

for _ in range(epochs):
    theta = theta - lr * gradient(theta, X_train, y_train)
    risks.append(risk(theta, X_train, y_train))
    acc_train.append(accuracy(theta, X_train, y_train))
    acc_test.append(accuracy(theta, X_test, y_test))

print(f"\nTask 2 done.")
print(f"  Final risk: {risks[-1]:.4f}")
print(f"  Final train accuracy: {acc_train[-1]:.4f}")
print(f"  Final test  accuracy: {acc_test[-1]:.4f}")

# PLOT1: risk vs iterations
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(epochs + 1), risks, color='steelblue', linewidth=1.5)
ax.set_xlabel("Iteration")
ax.set_ylabel("Risk R(θ)")
ax.set_title("Task 2 – Risk vs. GD Iteration")
ax.axvline(0, color='gray', linestyle='--', linewidth=0.8, label='Init point')
ax.legend()
plt.tight_layout()
plt.savefig("PLOT1.pdf")
plt.close()
print("  Saved PLOT1.pdf")

# PLOT2: train accuracy vs iterations
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(epochs + 1), acc_train, color='darkorange', linewidth=1.5)
ax.axvline(0, color='gray', linestyle='--', linewidth=0.8, label='Init point')
ax.set_xlabel("Iteration")
ax.set_ylabel("Accuracy")
ax.set_title("Task 2 – Training Accuracy vs. GD Iteration")
ax.legend()
plt.tight_layout()
plt.savefig("PLOT2.pdf")
plt.close()
print("  Saved PLOT2.pdf")

# PLOT3: test accuracy vs iterations
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(epochs + 1), acc_test, color='seagreen', linewidth=1.5)
ax.axvline(0, color='gray', linestyle='--', linewidth=0.8, label='Init point')
ax.set_xlabel("Iteration")
ax.set_ylabel("Accuracy")
ax.set_title("Task 2 – Testing Accuracy vs. GD Iteration")
ax.legend()
plt.tight_layout()
plt.savefig("PLOT3.pdf")
plt.close()
print("  Saved PLOT3.pdf")


# =============================================================================
# 3  Task 3 (6 points)
#
# - Generate feature X_1 from the mixture of 3 Gaussian distributions:
#     0.25*N(-1, sd=0.2) + 0.25*N(1, sd=0.2) + 0.5*N(5, sd=0.2)
#   Then generate X_2 from N(0, 1). In addition, generate binary class
#   variable such that P(Y=1|X)=0.1 when X_1 < 3 and P(Y=1|X)=0.9 when
#   X_1 >= 3. Generate variables X_1, X_2, Y of size n=1000.
# - Mutual information between binary Y and quantitative variable X is
#   defined as
#     I(Y, X) = H(X) - [P(Y=1)*H(X|Y=1) + P(Y=0)*H(X|Y=0)],
#   where entropy H(X) = -∫ f(x) log f(x) dx, with f being the true density
#   function of X.
# - Using Kernel density estimator and the samples generated in above items,
#   propose and compute estimators of I(Y, X_1) and I(Y, X_2). Print the
#   values of both estimators. Choose the optimal value of the smoothing
#   parameter for KDE.
# =============================================================================

n3 = 1000

# Generate X1 from mixture of 3 Gaussians
components = np.random.choice([0, 1, 2], size=n3, p=[0.25, 0.25, 0.50])
means = [-1, 1, 5]
X1 = np.array([np.random.normal(means[c], 0.2) for c in components])

# Generate X2 from N(0,1)
X2 = np.random.normal(0, 1, n3)

# Generate Y
p_y1 = np.where(X1 < 3, 0.1, 0.9)
Y3 = (np.random.rand(n3) < p_y1).astype(int)

def kde_entropy(samples, bw='scott'):
    """Estimate differential entropy H(X) = -∫ f(x) log f(x) dx via KDE."""
    kde = gaussian_kde(samples, bw_method=bw)
    # evaluate on a fine grid over the support
    x_min, x_max = samples.min() - 3 * samples.std(), samples.max() + 3 * samples.std()
    grid = np.linspace(x_min, x_max, 2000)
    f_grid = kde(grid)
    # numerical integration using the trapezoidal rule; avoid log(0)
    f_pos = np.where(f_grid > 1e-300, f_grid, 1e-300)
    h = -np.trapezoid(f_pos * np.log(f_pos), grid)
    return h

def mutual_information_kde(X, Y, bw='scott'):
    """Estimate I(Y, X) using KDE-based entropy estimators."""
    H_X = kde_entropy(X, bw=bw)

    mask1 = Y == 1
    mask0 = Y == 0
    p1 = mask1.mean()
    p0 = mask0.mean()

    H_X_given_Y1 = kde_entropy(X[mask1], bw=bw) if mask1.sum() > 1 else 0.0
    H_X_given_Y0 = kde_entropy(X[mask0], bw=bw) if mask0.sum() > 1 else 0.0

    cond_H = p1 * H_X_given_Y1 + p0 * H_X_given_Y0
    return H_X - cond_H

# Scott's rule is the optimal bandwidth choice for KDE
I_X1 = mutual_information_kde(X1, Y3, bw='scott')
I_X2 = mutual_information_kde(X2, Y3, bw='scott')

print(f"\nTask 3 done.")
print(f"  I(Y, X1) = {I_X1:.4f}  (KDE, Scott's bandwidth)")
print(f"  I(Y, X2) = {I_X2:.4f}  (KDE, Scott's bandwidth)")


# =============================================================================
# Task from Image 2 (Task 3, 5 points – smoothing spline variant)
#
# - Generate X of size n=1000 from uniform distribution on [-10, 10].
# - Let f be the density function of the random variable distributed from a
#   mixture of three Gaussian distributions:
#     0.25*N(m=-5, sd=0.5) + 0.25*N(m=0, sd=0.5) + 0.5*N(m=5, sd=0.5)
# - Generate response variable Y such that
#     y_i = 10*f(x_i) + epsilon_i, for i=1,...,1000 and
#     epsilon_i ~ N(m=0, sd=0.2)
# - Use smoothing spline method with default value of smoothing parameter.
#   Draw scatter plot (x_i, y_i), the curves corresponding to f and estimated
#   f. Save the results in file PLOT4.pdf.
# =============================================================================

n4 = 1000
X4 = np.random.uniform(-10, 10, n4)

def mixture_density(x):
    """Density of 0.25*N(-5,0.5) + 0.25*N(0,0.5) + 0.5*N(5,0.5)."""
    from scipy.stats import norm
    return (0.25 * norm.pdf(x, -5, 0.5)
            + 0.25 * norm.pdf(x, 0, 0.5)
            + 0.50 * norm.pdf(x, 5, 0.5))

f_X4 = mixture_density(X4)
eps = np.random.normal(0, 0.2, n4)
Y4 = 10 * f_X4 + eps

# Smoothing spline (UnivariateSpline with default smoothing)
sort_idx = np.argsort(X4)
X4_sorted = X4[sort_idx]
Y4_sorted = Y4[sort_idx]

spline = UnivariateSpline(X4_sorted, Y4_sorted)  # default smoothing parameter
x_grid = np.linspace(-10, 10, 1000)
f_true_grid = 10 * mixture_density(x_grid)
f_est_grid = spline(x_grid)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(X4, Y4, s=5, alpha=0.4, color='lightslategray', label='Data $(x_i, y_i)$')
ax.plot(x_grid, f_true_grid, color='royalblue', linewidth=2.0, label='True $10 \\cdot f(x)$')
ax.plot(x_grid, f_est_grid, color='crimson', linewidth=2.0, linestyle='--', label='Smoothing spline $\\hat{f}$')
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Task 3 (Image 2) – Smoothing Spline vs True Density Function")
ax.legend()
plt.tight_layout()
plt.savefig("PLOT4.pdf")
plt.close()
print(f"\nSpline task done.")
print("  Saved PLOT4.pdf")

Task 1 done.
  Training set: (1000, 10), class balance: 0.529
  Testing set:  (1000, 10), class balance: 0.515

Task 2 done.
  Final risk: 0.4816
  Final train accuracy: 0.9230
  Final test  accuracy: 0.9270
  Saved PLOT1.pdf
  Saved PLOT2.pdf
  Saved PLOT3.pdf

Task 3 done.
  I(Y, X1) = 0.6034  (KDE, Scott's bandwidth)
  I(Y, X2) = -0.0035  (KDE, Scott's bandwidth)

Spline task done.
  Saved PLOT4.pdf
